## Tools ##

In [1]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3-32b")
response = model.invoke("What is 2 + 2?")
response

AIMessage(content='<think>\nOkay, the user is asking "What is 2 + 2?" That\'s a basic arithmetic question. Let me think about how to approach this.\n\nFirst, I need to confirm that the question is straightforward. Adding 2 and 2 is pretty simple. The answer should be 4. But maybe the user is testing if I can handle basic math, or perhaps they want to see if there\'s any trick to it. Sometimes people ask this as a joke, but I don\'t think that\'s the case here.\n\nWait, in some contexts, like in certain books or movies, 2 + 2 might not equal 4 for dramatic effect, like in George Orwell\'s "1984" where the government tries to enforce that 2 + 2 = 5. But unless specified, the standard answer is 4. The user hasn\'t mentioned any such context, so I should stick with the mathematical answer.\n\nLet me also consider if there\'s any other way to interpret the question. In different number bases, could 2 + 2 be different? For example, in base 3, 2 + 2 would be 11 (which is 4 in decimal). But th

In [3]:
from langchain.tools import tool

@tool
def get_weather(location:str) ->str:
    """Get the weather at a location"""
    return f"It's rainy in{location}"

model_with_tools = model.bind_tools([get_weather])


In [4]:
response = model_with_tools.invoke("What's the weather in New Zealand?")
print(response)
for tool_call in response.tool_calls:
    print(f"Tools:{tool_call['name']}")
    print(f"Args:{tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in New Zealand. Let me check the tools provided. There\'s a function called get_weather that takes a location parameter. New Zealand is a location, so I need to call that function with "New Zealand" as the argument. I\'ll make sure the JSON is correctly formatted with the name and arguments. No other tools are available, so this should be the only function call needed.\n', 'tool_calls': [{'id': 'dx6xctydv', 'function': {'arguments': '{"location":"New Zealand"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 111, 'prompt_tokens': 154, 'total_tokens': 265, 'completion_time': 0.174656792, 'completion_tokens_details': {'reasoning_tokens': 86}, 'prompt_time': 0.006326399, 'prompt_tokens_details': None, 'queue_time': 0.159328632, 'total_time': 0.180983191}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_2bfcc54d36', 'service_tier': 'o

## Tool Execution loops ###

In [6]:
# step 1 model generates tool calls
messages = [{"role":"user","content":"What's the weather in Newyork?"}]
ai_msg = model_with_tools.invoke(messages)

#step 2 execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # executing the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# step 3 pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)



The weather in New York is rainy.


In [7]:
messages

[{'role': 'user', 'content': "What's the weather in Newyork?"},
 ToolMessage(content="It's rainy inNew York", name='get_weather', tool_call_id='74hh3az25')]